# Notebook 5a: Null Model Simulation (No Population Weighting)

This notebook constructs a gravity-based null model for counterfactual evacuation destination simulation **without** population weighting, as a robustness check.

**Paper reference:** Supplementary analysis — robustness of null model.

**Inputs:** `evacuees_for_regs.csv`, `tl_2019_08_bg/` shapefile

**Outputs:** Simulated destinations, similarity/connectedness comparisons


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns

In [ ]:
import itertools
from shapely.geometry import Point
from geopy.distance import geodesic

In [ ]:
evacuees = pd.read_csv("evacuess_for_regs.csv")
evacuees.head()

In [ ]:
evacuees.columns

In [ ]:
shapefile_path = 'tl_2019_08_bg'
cbg_geo = gpd.read_file(shapefile_path)

cbg_geo['GEOID'] = cbg_geo['GEOID'].str.lstrip('0')

columns_keep = ["GEOID", "geometry"]
colorado_cbgs = cbg_geo[columns_keep]

colorado_cbgs.head()

In [ ]:
colorado_cbgs['centroid'] = colorado_cbgs['geometry'].centroid

In [ ]:
cbgs = colorado_cbgs[['GEOID', 'centroid']]

# All possible pairs 
this takes time do not run it just read the saves csv - pairwise in the next section

In [ ]:
pairs = list(itertools.product(cbgs['GEOID'], cbgs['GEOID']))

In [ ]:
pairwise = pd.DataFrame(pairs, columns=['origin', 'destination'])

In [ ]:
pairwise = pairwise.merge(cbgs, left_on='origin', right_on='GEOID').rename(columns={'centroid': 'origin_centroid'})
pairwise = pairwise.merge(cbgs, left_on='destination', right_on='GEOID').rename(columns={'centroid': 'destination_centroid'})


In [ ]:
pairwise['distance'] = pairwise.apply(
    lambda row: geodesic(
        (row['origin_centroid'].y, row['origin_centroid'].x),
        (row['destination_centroid'].y, row['destination_centroid'].x)
    ).meters, axis=1
)

In [ ]:
pairwise.head()

In [ ]:
pairwise.to_csv('pairwise')

# Calculating homophily and similarity For all OD combinations

In [ ]:
pairwise = pd.read_csv('pairwise')

In [ ]:
pairwise.head()

In [ ]:
census = pd.read_csv("Shapefile_census/ACS_Colorado_2019_Block_Groups.csv")
census.columns

In [ ]:
census['white_population'] = (census['white_population']  /census['total_population'] )*100
census['black_population'] = (census['black_population']  /census['total_population'] )*100
census['asian_population'] = (census['asian_population']  /census['total_population'] )*100
census['education_atleast_one_degree'] = (census['total_population_25_over']  /census['total_population'] )*100

In [ ]:
columns_keep = ['total_population','total_housing_units','white_population', 'black_population', 'asian_population', 'education_atleast_one_degree', 'unemployment_rate','median_household_income', 'geoid' ]
census = census[columns_keep]
census.head()

In [ ]:
census['geoid'] = census['geoid'].astype(str)
cbg_geo['GEOID'] = cbg_geo['GEOID'].astype(str)

In [ ]:
census_r1 = pd.merge(census, cbg_geo, 
                    left_on='geoid', right_on='GEOID', 
                    how='left').rename(columns={'geometry': 'geometry'})

In [ ]:
census_r1 = gpd.GeoDataFrame(census_r1, geometry='geometry')

In [ ]:
census_r1['area_sq_km'] = census_r1['geometry'].area / 10**6
# Calculate population density (people per square kilometer) - or
census_r1['population_density'] = census_r1['total_population'] / census_r1['area_sq_km']

In [ ]:
census_r1.head()

In [ ]:
census_D1 = census_r1.rename(
    columns=lambda x: 'Or_' + x  if x != 'geoid' else x
)
census_D1.head()

In [ ]:
census_D1['geoid'] = census_D1['geoid'].astype(str)
pairwise['origin'] = pairwise['origin'].astype(str)
pairwise_census = pairwise.merge(census_D1, left_on = "origin", right_on = "geoid", how = "inner")

In [ ]:
census_D2 = census_r1.rename(
    columns=lambda x: 'D1_' + x #if x != 'geoid' else x
)
census_D2.head()

In [ ]:
census_D2['D1_geoid'] = census_D2['D1_geoid'].astype(str)
pairwise_census['destination'] = pairwise_census['destination'].astype(str)
pairwise_census_od = pairwise_census.merge(census_D2, left_on = "destination", right_on = "D1_geoid", how = "inner")

In [ ]:
pairwise_census_od.columns

In [ ]:
# first_100_origins = pairwise_census_od['origin'].unique()[:5]
# filtered_pairwise_census_od = pairwise_census_od[pairwise_census_od['origin'].isin(first_100_origins)]
# filtered_pairwise_census_od.to_csv("pairwise_census_od_trial")

In [ ]:
# pairwise_census_od = pd.read_csv("pairwise_census_od_trial")
# pairwise_census_od.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
origin_columns = [
       #'Or_population_density', 
       'Or_white_population', 'Or_black_population',
       'Or_asian_population', 'Or_education_atleast_one_degree',
       'Or_unemployment_rate', 
       'Or_median_household_income'	   
]

destination_columns = [
        #'D1_population_density', 
        'D1_white_population',
       'D1_black_population', 'D1_asian_population',
       'D1_education_atleast_one_degree', 'D1_unemployment_rate',
       'D1_median_household_income'
]

pairwise_census_od = pairwise_census_od.dropna()
# Normalize the data for origin and destination separately
scaler = MinMaxScaler()
normalized_origin = scaler.fit_transform(pairwise_census_od[origin_columns])
normalized_destination = scaler.fit_transform(pairwise_census_od[destination_columns])

# Compute cosine similarity for each row
homophily_scores = []
for i in range(len(pairwise_census_od)):
    origin_vector = normalized_origin[i].reshape(1, -1)
    destination_vector = normalized_destination[i].reshape(1, -1)
    similarity = cosine_similarity(origin_vector, destination_vector)[0][0]
    homophily_scores.append(similarity)

# Append the homophily scores to the original DataFrame
pairwise_census_od['Similarity'] = homophily_scores
pairwise_census_od.head()

In [ ]:
#pairwise_census_od.to_csv('pairwise_census_od')

In [ ]:
SCI = pd.read_csv("colorado_SCI_ZIP_CBG.csv")

In [ ]:
duplicates = SCI.duplicated(subset=['cbg_user', 'cbg_fr'], keep=False)
if duplicates.sum() > 0:
    print(f"There are {duplicates.sum()} duplicate rows in SCI based on 'cbg_user' and 'cbg_fr'.")

SCI_unique = SCI.groupby(['cbg_user', 'cbg_fr'], as_index=False)['scaled_sci'].mean()

SCI_unique = SCI.groupby(['cbg_user', 'cbg_fr'], as_index=False)['scaled_sci'].mean()
print(f"Shape of SCI after removing duplicates: {SCI_unique.shape}")
SCI_unique.head()

In [ ]:
SCI_unique['cbg_user'] = SCI_unique['cbg_user'].astype(str)
SCI_unique['cbg_fr'] = SCI_unique['cbg_fr'].astype(str)
pairwise_census_od['origin'] = pairwise_census_od['origin'].astype(str)
pairwise_census_od['destination'] = pairwise_census_od['destination'].astype(str)

In [ ]:
pairwise_census_od_SCI = pd.merge(
    pairwise_census_od, 
    SCI_unique, 
    left_on=['origin', 'destination'],
    right_on=['cbg_user', 'cbg_fr'], 
    how='left' 
    )
pairwise_census_od_SCI.head()

In [ ]:
columns = ['origin', 'destination', 'distance', 'Similarity','scaled_sci']
pairwise_subset = pairwise_census_od_SCI[columns]

In [ ]:
#pairwise_census_od_SCI.to_csv('pairwise_subset.csv.gz')

In [ ]:
pairwise_subset.head()

# Evacuees subset 

In [ ]:
# Define the bin edges and labels as you did
bin_edges = np.arange(0, evacuees['distance_OD'].max() + 20000, 20000)
bin_labels = [f'{int(left)}-{int(right)}' for left, right in zip(bin_edges[:-1], bin_edges[1:])]

# Ensure bins are left-inclusive and avoid losing boundary values with right=False
evacuees['distance_range'] = pd.cut(evacuees['distance_OD'], bins=bin_edges, labels=bin_labels, include_lowest=True, right=False)

# Convert distance_range to string, split to get min and max distance values
evacuees['distance_range'] = evacuees['distance_range'].astype(str)
evacuees['distance_min'] = evacuees['distance_range'].apply(lambda x: int(x.split('-')[0]))
evacuees['distance_max'] = evacuees['distance_range'].apply(lambda x: int(x.split('-')[1]))
evacuees.head()

In [ ]:
evacuees_subset = evacuees[['Unnamed: 0', 'pre_crisis_home_cbg', 'crisis_home_cbg', 'D1_total_population', 'distance_OD', 'distance_range', 'distance_min',	'distance_max']]
evacuees_subset.head()

# Merging evacuees with pairwise

In [ ]:
#pairwise_subset = pd.read_csv('pairwise_subset.csv.gz')

In [ ]:
evacuees_subset['pre_crisis_home_cbg'] = evacuees_subset['pre_crisis_home_cbg'].astype(str)
pairwise_subset['origin'] = pairwise_subset['origin'].astype(str)

In [ ]:
evacuees_subset['crisis_home_cbg'] = evacuees_subset['crisis_home_cbg'].astype(str)
pairwise_subset['destination'] = pairwise_subset['destination'].astype(str)


In [ ]:
merged_df = evacuees_subset.merge(pairwise_subset, left_on='pre_crisis_home_cbg', right_on='origin', how = 'left')

merged_df.head()

In [ ]:
# Now, modify the filtering step to ensure the destination bin is not lost
filtered_df = merged_df[(merged_df['distance_OD'] >= merged_df['distance_min']) & (merged_df['distance_OD'] <= merged_df['distance_max'])]

# Drop unnecessary columns after filtering
filtered_df = filtered_df.drop(['distance_min', 'distance_max'], axis=1)

# Check the filtered dataframe
filtered_df.head()

In [ ]:
census = pd.read_csv("Shapefile_census/ACS_Colorado_2019_Block_Groups.csv")
census.columns

In [ ]:
columns_keep = ['geoid','total_population']
census = census[columns_keep]
census.head()

In [ ]:
census['geoid'] = census['geoid'].astype(str)
filtered_df['destination'] = filtered_df['destination'].astype(str)

In [ ]:
filtered_df = filtered_df.merge(census, left_on = "destination", right_on = "geoid", how = "left")

In [ ]:
filtered_df.to_csv('for_simulation.csv.gz')

# Probability

## calculating probabilities for all pairs

In [ ]:
filtered_df['probabilities'] = filtered_df.groupby('Unnamed: 0')['total_population'].transform(lambda x: x / x.sum())
filtered_df.head()

In [ ]:
filtered_df.columns

In [ ]:
filtered_df.to_csv('simulated_final.csv.gz')

## Subset of actual destinations

In [ ]:
crisis_home_df = filtered_df[filtered_df['crisis_home_cbg'] == filtered_df['destination']]

In [ ]:
crisis_home_df = crisis_home_df.merge(census, left_on = "destination", right_on = "geoid", how = "left")

In [ ]:
crisis_home_df.shape

In [ ]:
crisis_home_df.head()

In [ ]:
crisis_home_df['probabilities_log'] = crisis_home_df['probabilities'].apply(np.log1p)

In [ ]:
crisis_home_df.to_csv("actual_location.csv.gz")

## Subset of random dest weighted by pop probabilities

In [ ]:
random_destination_pop_df = filtered_df.groupby('Unnamed: 0').apply(lambda x: x.sample(1, weights=x['probabilities'])).reset_index(drop=True)
random_destination_pop_df['probabilities_log'] = random_destination_pop_df['probabilities'].apply(np.log1p)
random_destination_pop_df.head()

In [ ]:
random_destination_pop_df.to_csv("simulated_location.csv.gz")

# PLOT FINAL

In [ ]:
actual_destination = pd.read_csv("actual_location.csv.gz")
actual_destination['Familiarity_log'] = actual_destination['scaled_sci'].apply(np.log1p)
actual_destination['Similarity_log'] = actual_destination['Similarity'].apply(np.log1p)
actual_destination.head()

In [ ]:
simulated_destination = pd.read_csv("simulated_location.csv.gz")
simulated_destination['Familiarity_log'] = simulated_destination['scaled_sci'].apply(np.log1p)
simulated_destination['Similarity_log'] = simulated_destination['Similarity'].apply(np.log1p)
simulated_destination.head()

In [ ]:
plt.figure(figsize=(7, 3))

sns.histplot(x='Familiarity_log', data=actual_destination, label='Actual Destinations', color='orange', fill=True)
sns.histplot(x='Familiarity_log', data=simulated_destination, label='Random Destination', color='teal', fill=True)
mean_actual = actual_destination['Familiarity_log'].mean()
mean_random = simulated_destination['Familiarity_log'].mean()

plt.axvline(mean_actual, color='orange', linestyle='--', label='Mean Actual Dest.', linewidth=1)
plt.axvline(mean_random, color='teal', linestyle='--', label='Mean Random Dest.', linewidth=1)
#plt.xlim(0.9, 1.2)
plt.yscale('log')
plt.xlabel('Familiarity')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 3))

sns.histplot(x='Similarity_log', data=simulated_destination, label='Random Destination', color='teal', fill=True)
sns.histplot(x='Similarity_log', data=actual_destination, label='Actual Destinations', color='orange', fill=True)
mean_actual = actual_destination['Similarity_log'].mean()
mean_random = simulated_destination['Similarity_log'].mean()

plt.axvline(mean_actual, color='orange', linestyle='--', label='Mean Actual Dest.', linewidth=1)
plt.axvline(mean_random, color='teal', linestyle='--', label='Mean Random Dest.', linewidth=1)
#plt.xlim(0.9, 1.2)
plt.yscale('log')
plt.xlabel('Similarity')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
combined_act_sim = pd.concat([
    actual_destination.assign(destination_type='Actual Destinations'),
    simulated_destination.assign(destination_type='Random Destinations')
])
combined_act_sim = combined_act_sim.sort_values(by='Unnamed: 0.1').reset_index(drop=True)

combined_act_sim.head()

In [ ]:
actual_data_sim = combined_act_sim[combined_act_sim['destination_type'] == 'Actual Destinations']['Similarity_log']
random_data_sim = combined_act_sim[combined_act_sim['destination_type'] == 'Random Destinations']['Similarity_log']

In [ ]:
actual_data_sim_mean = np.mean(actual_data_sim)
random_data_sim_mean = np.mean(random_data_sim)
actual_data_sim_median = np.median(actual_data_sim)
random_data_sim_median = np.median(random_data_sim)

print(f"Actual Destinations: {actual_data_sim_mean, actual_data_sim_median}")
print(f"Random Destinations: {random_data_sim_mean, random_data_sim_median}")

In [ ]:
plt.figure(figsize=(5, 3))
box = plt.boxplot([actual_data_sim, random_data_sim], patch_artist=True, vert=False, widths=0.6, labels=['Actual Destinations', 'Random Destinations'])

colors = ['steelblue', 'steelblue']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('none')  
    
for median, color in zip(box['medians'], colors):
    median_value = median.get_xdata()[0]  # Get the median value position
    plt.axvline(median_value, color=color, linestyle='--', linewidth=1)  # Vertical line at the median

for flier in box['fliers']:
    flier.set(marker='')

plt.xlim(0.6, 0.8)

plt.tight_layout()
plt.show()
plt.savefig('boxplot_similarity.png', dpi=300)

In [ ]:
plt.figure(figsize=(5, 3))
box = plt.boxplot([actual_data_sim, random_data_sim], patch_artist=True, vert=False, widths=0.6, labels=['Actual Destinations', 'Random Destinations'])

colors = ['steelblue', 'steelblue']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('none')  
    
for median, color in zip(box['medians'], colors):
    median_value = median.get_xdata()[0]  # Get the median value position
    plt.axvline(median_value, color=color, linestyle='--', linewidth=1)  # Vertical line at the median

for flier in box['fliers']:
    flier.set(marker='')

plt.xlim(0.6, 0.75)

plt.tight_layout()

# Force the plot to render before saving
plt.draw()

# Save the plot in 300 DPI
plt.savefig('boxplot_Similarity.png', dpi=300)

plt.show()

In [ ]:
actual_data_fam = combined_act_sim[combined_act_sim['destination_type'] == 'Actual Destinations']['Familiarity_log']
random_data_fam = combined_act_sim[combined_act_sim['destination_type'] == 'Random Destinations']['Familiarity_log']

In [ ]:
actual_data_sim_mean = np.mean(actual_data_fam)
random_data_sim_mean = np.mean(random_data_fam)
actual_data_sim_median = np.median(actual_data_fam)
random_data_sim_median = np.median(random_data_fam)

print(f"Actual Destinations: {actual_data_sim_mean, actual_data_sim_median}")
print(f"Random Destinations: {random_data_sim_mean, random_data_sim_median}")

In [ ]:
# Remove NaN values from both datasets
actual_data_fam_clean = actual_data_fam.dropna()
random_data_fam_clean = random_data_fam.dropna()

# Create a box plot using Matplotlib with cleaned data
plt.figure(figsize=(5, 3))

# Box plot with outliers shown and customized width
box = plt.boxplot([actual_data_fam_clean, random_data_fam_clean], patch_artist=True, vert=False, widths=0.6, labels=['Actual Destinations', 'Random Destinations'])

# Customizing the appearance of the box plot
colors = ['sandybrown', 'sandybrown']  # Different colors for actual and random

# Change box colors and remove the edge color (no boundary)
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('none')  # Remove the boundary

# Show the outliers clearly
for flier in box['fliers']:
    flier.set(marker='')  # Set outlier appearance

# Extend the median line vertically through the entire plot
for median, color in zip(box['medians'], colors):
    median_value = median.get_xdata()[0]  # Get the median value position
    plt.axvline(median_value, color=color, linestyle='--', linewidth=1)  # Vertical line at the median

# Set appropriate x-axis limits to fit the range of your data
plt.xlim(7.5, 20)  # Adjust based on the data range

# Add labels and title
plt.xlabel('Familiarity (Raw Values)')
plt.title('Box Plot of Familiarity for Actual and Random Destinations (Cleaned Data)')

plt.tight_layout()
plt.show()
plt.savefig('boxplot_familiarity.png', dpi=300)

In [ ]:
# Remove NaN values from both datasets
actual_data_fam_clean = actual_data_fam.dropna()
random_data_fam_clean = random_data_fam.dropna()

# Create a box plot using Matplotlib with cleaned data
plt.figure(figsize=(5, 3))

# Box plot with outliers shown and customized width
box = plt.boxplot([actual_data_fam_clean, random_data_fam_clean], patch_artist=True, vert=False, widths=0.6, labels=['Actual Destinations', 'Random Destinations'])

# Customizing the appearance of the box plot
colors = ['sandybrown', 'sandybrown']  # Different colors for actual and random

# Change box colors and remove the edge color (no boundary)
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('none')  # Remove the boundary

# Show the outliers clearly
for flier in box['fliers']:
    flier.set(marker='')  # Set outlier appearance

# Extend the median line vertically through the entire plot
for median, color in zip(box['medians'], colors):
    median_value = median.get_xdata()[0]  # Get the median value position
    plt.axvline(median_value, color=color, linestyle='--', linewidth=1)  # Vertical line at the median

# Set appropriate x-axis limits to fit the range of your data
plt.xlim(7.5, 20)  # Adjust based on the data range

# Add labels and title
plt.xlabel('Familiarity (Raw Values)')
plt.title('Box Plot of Familiarity for Actual and Random Destinations (Cleaned Data)')

plt.tight_layout()

# Force the plot to render before saving
plt.draw()

# Save the plot in 300 DPI
plt.savefig('boxplot_familiarity.png', dpi=300)

plt.show()

<!-- ## import seaborn as sns

# Create a KDE plot for both datasets
plt.figure(figsize=(7, 3))

# KDE plot for actual destinations
sns.kdeplot(actual_data_fam, label='Actual Destinations', color='orange', fill=True)

# KDE plot for random destinations
sns.kdeplot(random_data_fam, label='Random Destinations', color='teal', fill=True)

# Add labels and title
plt.xlabel('Connectedness (Raw Values)')
plt.ylabel('Density')
plt.title('KDE Plot of Connectedness for Actual and Random Destinations')

# Add a legend
plt.legend()

# Show the plot
plt.tight_layout()
plt.show() -->

In [ ]:
import matplotlib.pyplot as plt

# Create a histogram for actual and random destinations
plt.figure(figsize=(7, 3))

# Plot histogram for actual destinations
plt.hist(actual_data_fam, bins=10, alpha=0.7, label='Actual Destinations', color='orange')

# Plot histogram for random destinations
plt.hist(random_data_fam, bins=10, alpha=0.7, label='Random Destinations', color='teal')

# Add labels and title
plt.xlabel('Familiarity (Raw Values)')
plt.ylabel('Frequency')
plt.title('Histogram of Familiarity for Actual and Random Destinations')

# Add a legend to differentiate the two distributions
plt.legend()

# Display the plot
plt.tight_layout()
plt.show()

# Connectedness and Similarity plot

## Actual subset

In [ ]:
crisis_home_df['pre_crisis_home_cbg'] = crisis_home_df['pre_crisis_home_cbg'].astype(str)
crisis_home_df['crisis_home_cbg'] = crisis_home_df['crisis_home_cbg'].astype(str)

pairwise_census_od_SCI['origin'] = pairwise_census_od_SCI['origin'].astype(str)
pairwise_census_od_SCI['destination'] = pairwise_census_od_SCI['destination'].astype(str)

actual_dest_FS = crisis_home_df.merge(
    pairwise_census_od_SCI, 
    left_on=['pre_crisis_home_cbg', 'crisis_home_cbg'], 
    right_on=['origin', 'destination'], 
    how='left'
).rename(columns={'Similarity': 'Act_Similarity', 'scaled_sci': 'Act_Familiarity'})

actual_dest_FS.head()

In [ ]:
actual_dest_FS.shape

In [ ]:
actual_dest_FS.to_csv("actual_destinations.csv")

## Highest probability subset

In [ ]:
highest_prob_df['pre_crisis_home_cbg'] = random_destination_df['pre_crisis_home_cbg'].astype(str)
highest_prob_df['destination'] = highest_prob_df['destination'].astype(str)

pairwise_census_od_SCI['origin'] = pairwise_census_od_SCI['origin'].astype(str)
pairwise_census_od_SCI['destination'] = pairwise_census_od_SCI['destination'].astype(str)

highest_prob_df_FS = highest_prob_df.merge(
    pairwise_census_od_SCI, 
    left_on=['pre_crisis_home_cbg', 'destination'], 
    right_on=['origin', 'destination'], 
    how='inner'
).rename(columns={'Similarity': 'Hp_Similarity', 'scaled_sci': 'Hp_Familiarity'})

highest_prob_df_FS.head()

In [ ]:
highest_prob_df_FS.to_csv("highest_prob_destinations.csv")

## Random subset

In [ ]:
random_destination_df['pre_crisis_home_cbg'] = random_destination_df['pre_crisis_home_cbg'].astype(str)
random_destination_df['destination'] = random_destination_df['destination'].astype(str)

pairwise_census_od_SCI['origin'] = pairwise_census_od_SCI['origin'].astype(str)
pairwise_census_od_SCI['destination'] = pairwise_census_od_SCI['destination'].astype(str)

random_destination_df_FS = random_destination_df.merge(
    pairwise_census_od_SCI, 
    left_on=['pre_crisis_home_cbg', 'destination'], 
    right_on=['origin', 'destination'], 
    how='inner'
).rename(columns={'Similarity': 'Rp_Similarity', 'scaled_sci': 'Rp_Familiarity'})

random_destination_df_FS.head()

In [ ]:
random_destination_df_FS.to_csv("random_destinations.csv")

## Random weighted subset

In [ ]:
random_destination_pop_df['pre_crisis_home_cbg'] = random_destination_pop_df['pre_crisis_home_cbg'].astype(str)
random_destination_pop_df['destination'] = random_destination_pop_df['destination'].astype(str)

pairwise_census_od_SCI['origin'] = pairwise_census_od_SCI['origin'].astype(str)
pairwise_census_od_SCI['destination'] = pairwise_census_od_SCI['destination'].astype(str)

random_destination_pop_df_FS = random_destination_pop_df.merge(
    pairwise_census_od_SCI, 
    left_on=['pre_crisis_home_cbg', 'destination'], 
    right_on=['origin', 'destination'], 
    how='inner'
).rename(columns={'Similarity': 'Rwp_Similarity', 'scaled_sci': 'Rwp_Familiarity'})

random_destination_pop_df_FS.head()

In [ ]:
random_destination_pop_df_FS.to_csv("random_destinations_pop_weighted.csv")

In [ ]:
actual_destinations=pd.read_csv("actual_location.csv.gz")

In [ ]:
actual_destinations.shape

In [ ]:
simulated_destinations = pd.read_csv("simulated_location.csv.gz")

In [ ]:
# highest_prob_destinations['source'] = 'highest_probability'
# actual_destinations['source'] = 'actual_destination'
# random_destinations['source'] = 'random_destination'
# random_destinations_pop_weighted['source'] = 'random_destination'

In [ ]:
# df = pd.concat([highest_prob_df_FS, crisis_home_df_FS,random_destination_df_FS ]).drop_duplicates().reset_index(drop=True)
# final_df = df.sort_values(by='Unnamed: 0').reset_index(drop=True)
# final_df.head()

In [ ]:
# final_df.to_csv("for_probablity_plot.csv")

## distribution of probabilities

In [ ]:
plt.figure(figsize=(3, 4))

#sns.kdeplot(random_destinations['probabilities_log'], label='Random Probability Destination', color='orange', fill=True)
#sns.kdeplot(highest_prob_destinations['probabilities_log'], label='Highest Probability Destination', color='orange', fill=True)
sns.kdeplot(actual_destinations['probabilities_log'], label='Crisis Home CBG', color='orange', fill=True)
sns.kdeplot(random_destinations_pop_weighted['probabilities_log'], label='Random weighted Probability Destination', color='teal', fill=True)

#plt.title('Distribution of Probabilities')
plt.xlabel('Probability')
plt.ylabel('Density')
#plt.legend()
plt.show()

## plotting connectedness and similarity

In [ ]:
#highest_prob_destinations['Hp_Similarity_log'] = highest_prob_destinations['Hp_Similarity'].apply(np.log1p)
highest_prob_destinations['Hp_Familiarity_log'] = highest_prob_destinations['Hp_Familiarity'].apply(np.log1p)
highest_prob_destinations['Hp_Similarity_log'] = highest_prob_destinations['Hp_Similarity'].apply(lambda x: np.log10(x + 1))
#actual_destinations['Act_Similarity_log'] = actual_destinations['Act_Similarity'].apply(np.log1p)
actual_destinations['Act_Similarity_log'] = actual_destinations['Act_Similarity'].apply(lambda x: np.log10(x + 1))
actual_destinations['Act_Familiarity_log'] = actual_destinations['Act_Familiarity'].apply(np.log1p)
#random_destinations['Rp_Similarity_log'] = random_destinations['Rp_Familiarity'].apply(np.log1p)
random_destinations['Rp_Similarity_log'] = random_destinations['Rp_Similarity'].apply(lambda x: np.log10(x + 1))
random_destinations['Rp_Familiarity_log'] = random_destinations['Rp_Familiarity'].apply(np.log1p)
#random_destinations_pop_weighted['Rwp_Similarity_log'] = random_destinations_pop_weighted['Rwp_Similarity'].apply(np.log1p)
random_destinations_pop_weighted['Rwp_Similarity_log'] = random_destinations_pop_weighted['Rwp_Similarity'].apply(lambda x: np.log10(x + 1))
random_destinations_pop_weighted['Rwp_Familiarity_log'] = random_destinations_pop_weighted['Rwp_Familiarity'].apply(np.log1p)

In [ ]:
actual_destinations.head()

In [ ]:
plt.figure(figsize=(5, 3))

plt.subplot(1, 2, 1)
#sns.ecdfplot(x='Hp_Similarity_log', data=highest_prob_destinations, label='Highest Probability', color='orange')
sns.ecdfplot(x='Act_Similarity_log', data=actual_destinations, label='Actual Destinations', color='orange')
#sns.ecdfplot(x='Rp_Similarity', data=random_destinations, label='Random Destination', color='red')
sns.ecdfplot(x='Rwp_Similarity_log', data=random_destinations_pop_weighted, label='Random Destination (Pop-Weighted)', color='teal')
plt.xlabel('Similarity')
plt.ylabel('Cumulative Probability')
#plt.legend()

plt.subplot(1, 2, 2)
#sns.ecdfplot(x='Hp_Familiarity_log', data=highest_prob_destinations, label='Highest Probability', color='orange')
sns.ecdfplot(x='Act_Familiarity_log', data=actual_destinations, label='Actual Destinations', color='orange')
#sns.ecdfplot(x='Rp_Familiarity', data=random_destinations, label='Random Destination', color='red')
sns.ecdfplot(x='Rwp_Familiarity_log', data=random_destinations_pop_weighted, label='Random Destination (Pop-Weighted)', color='teal')
plt.xlabel('Familiarity')
plt.ylabel('Cumulative Probability')
#plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 3))

sns.histplot(x='Act_Similarity', data=actual_destinations, label='Actual Destinations', color='orange', fill=True)
sns.histplot(x='Rwp_Similarity', data=random_destinations_pop_weighted, label='Random Destination', color='teal', fill=True)
mean_actual = actual_destinations['Act_Similarity'].mean()
mean_random = random_destinations_pop_weighted['Rwp_Similarity'].mean()

plt.axvline(mean_actual, color='orange', linestyle='--', label='Mean Actual Dest.', linewidth=1)
plt.axvline(mean_random, color='teal', linestyle='--', label='Mean Random Dest.', linewidth=1)
#plt.xlim(0.9, 1.2)
plt.yscale('log')
plt.xlabel('Similarity')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

# T-tests

In [ ]:
mean_actual = evacuees['homophily'].mean()
mean_random = random_destinations_pop_weighted['Rwp_Similarity'].mean()
std_actual = evacuees['homophily'].std()
std_random = random_destinations_pop_weighted['Rwp_Similarity'].std()

mean_distance = abs(mean_actual - mean_random)

n_actual = len(evacuees)
n_random = len(random_destinations_pop_weighted)

t_stats = (mean_actual - mean_random) / np.sqrt((std_actual**2 / n_actual) + (std_random**2 / n_random))

print(t_stats)

In [ ]:
plt.figure(figsize=(4, 3))

sns.kdeplot(x='Act_Familiarity_log', data=actual_destinations, label='Actual Destinations', color='orange', fill=True)
sns.kdeplot(x='Rwp_Familiarity_log', data=random_destinations_pop_weighted, label='Random Destination', color='teal', fill=True)

mean_actual = actual_destinations['Act_Familiarity_log'].mean()
mean_random = random_destinations_pop_weighted['Rwp_Familiarity_log'].mean()

plt.axvline(mean_actual, color='darkgrey', linestyle='--', label='Mean Actual Dest.', linewidth=1)
plt.axvline(mean_random, color='teal', linestyle='--', label='Mean Random Dest.', linewidth=1)

plt.xlabel('Familiarityy')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
mean_actual = actual_destinations['Act_Familiarity'].mean()
mean_random = random_destinations_pop_weighted['Rwp_Familiarity'].mean()
std_actual = actual_destinations['Act_Familiarity'].std()
std_random = random_destinations_pop_weighted['Rwp_Familiarity'].std()

mean_distance = abs(mean_actual - mean_random)

n_actual = len(actual_destinations)
n_random = len(random_destinations_pop_weighted)

t_stats = (mean_actual - mean_random) / np.sqrt((std_actual**2 / n_actual) + (std_random**2 / n_random))

print(t_stats)

# Distance Plot

In [ ]:
actual_destination.head()

In [ ]:
actual_destination['log_distance_OD'] = actual_destination['distance_OD'].apply(np.log1p)

In [ ]:
from matplotlib.colors import Normalize

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Assign the actual column names to variables
distance_var = 'log_distance_OD'
homophily_var = 'Similarity_log'
sci_var = 'Familiarity_log'

# Bin the distance data into 10 equal-width bins
actual_destination['distance_bin'] = pd.cut(actual_destination[distance_var], bins=10, labels=False)

# Function to plot box plots with a straight trend line (linear regression)
def plot_boxplot_with_trend(data, variable, title, color, filename=None):
    plt.figure(figsize=(6, 4))

    # Create the boxplot
    ax = sns.boxplot(x='distance_bin', y=variable, data=data, showfliers=False,
                     medianprops=dict(color='black'), palette=[color] * 10)
    
    # Add the straight trend line using sns.regplot (without scatter points)
    sns.regplot(x='distance_bin', y=variable, data=data, scatter=False, 
                color='darkgrey', line_kws={'linewidth': 2}, ax=ax)
    
    # Set plot labels and titles
    plt.title(title)
    plt.xlabel('Binned Distance (10 equal bins)')
    plt.ylabel(variable)

    # Save the plot if filename is provided
    if filename:
        plt.savefig(filename, dpi=300)

    plt.show()

# Plot for homophily vs. binned distance with blue color and save the plot
plot_boxplot_with_trend(actual_destination, homophily_var, 
                        f'Boxplot with Trend Line: {homophily_var} vs Binned Distance', 
                        color='steelblue', filename='homophily_vs_distance.png')

# Plot for SCI vs. binned distance with orange color and save the plot
plot_boxplot_with_trend(actual_destination, sci_var, 
                        f'Boxplot with Trend Line: {sci_var} vs Binned Distance', 
                        color='chocolate', filename='sci_vs_distance.png')

In [ ]:
print(mean_homophily_per_bin)
print(mean_sci_per_bin)